In [ ]:
# ============================================================
# SPEECH-BASED COMPLAINT ANALYSIS SYSTEM
# GOOGLE COLAB - SINGLE CELL PROJECT
# ============================================================

# Install Flask
!pip -q install flask

import os
import sqlite3
import threading
import time

from flask import Flask, request, jsonify, render_template_string
from google.colab import output


# ============================================================
# FLASK APPLICATION
# ============================================================

app = Flask(__name__)

DATABASE = "complaints.db"


# ============================================================
# DATABASE INITIALIZATION
# ============================================================

def init_db():

    conn = sqlite3.connect(DATABASE)

    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS complaints (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            complaint TEXT NOT NULL,
            category TEXT NOT NULL,
            priority TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()


# ============================================================
# COMPLAINT ANALYSIS
# ============================================================

def analyze_complaint(text):

    text = text.lower()

    # ---------------- CATEGORY ----------------

    if any(word in text for word in [
        "bus",
        "transport",
        "driver",
        "vehicle",
        "college bus"
    ]):
        category = "Transport"

    elif any(word in text for word in [
        "hostel",
        "room",
        "warden",
        "mess",
        "food",
        "water"
    ]):
        category = "Hostel"

    elif any(word in text for word in [
        "teacher",
        "class",
        "exam",
        "marks",
        "attendance",
        "subject",
        "faculty"
    ]):
        category = "Academics"

    elif any(word in text for word in [
        "wifi",
        "internet",
        "computer",
        "lab",
        "building",
        "fan",
        "ac",
        "electricity"
    ]):
        category = "Infrastructure"

    else:
        category = "Other"


    # ---------------- PRIORITY ----------------

    if any(word in text for word in [
        "urgent",
        "emergency",
        "danger",
        "critical",
        "immediately",
        "serious"
    ]):
        priority = "High"

    elif any(word in text for word in [
        "problem",
        "issue",
        "not working",
        "complaint",
        "broken",
        "poor"
    ]):
        priority = "Medium"

    else:
        priority = "Low"


    return category, priority


# ============================================================
# HTML PAGE
# ============================================================

HTML = """

<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>Speech Complaint Analysis System</title>


<style>

* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}


body {

    font-family: Arial, sans-serif;

    background:
        linear-gradient(
            135deg,
            #eef2ff,
            #f8fafc
        );

    min-height: 100vh;

    color: #1e293b;

}


.container {

    width: 92%;

    max-width: 1100px;

    margin: auto;

    padding: 35px 0;

}


.header {

    text-align: center;

    margin-bottom: 30px;

}


.header h1 {

    font-size: 32px;

    color: #263b80;

    margin-bottom: 10px;

}


.header p {

    color: #64748b;

    font-size: 16px;

}


.card {

    background: white;

    border-radius: 18px;

    padding: 25px;

    margin-bottom: 25px;

    box-shadow:
        0 8px 30px
        rgba(0,0,0,0.08);

}


.card h2 {

    color: #263b80;

    margin-bottom: 18px;

}


textarea {

    width: 100%;

    height: 150px;

    resize: vertical;

    border: 2px solid #e2e8f0;

    border-radius: 12px;

    padding: 15px;

    font-size: 16px;

    outline: none;

}


textarea:focus {

    border-color: #263b80;

}


.buttons {

    display: flex;

    gap: 12px;

    margin-top: 15px;

    flex-wrap: wrap;

}


button {

    border: none;

    padding: 13px 20px;

    border-radius: 10px;

    cursor: pointer;

    font-size: 15px;

    font-weight: bold;

}


.record {

    background: #e74c3c;

    color: white;

}


.submit {

    background: #263b80;

    color: white;

}


.clear {

    background: #e2e8f0;

    color: #334155;

}


button:hover {

    opacity: 0.85;

}


.status {

    margin-top: 15px;

    font-weight: bold;

    color: #475569;

}


.result-grid {

    display: grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap: 15px;

}


.result {

    padding: 20px;

    border-radius: 12px;

    background: #f8fafc;

    border: 1px solid #e2e8f0;

}


.result h3 {

    color: #64748b;

    font-size: 14px;

    margin-bottom: 10px;

}


.result p {

    font-size: 18px;

    font-weight: bold;

}


.history-item {

    border-bottom: 1px solid #e2e8f0;

    padding: 18px 0;

}


.history-item:last-child {

    border-bottom: none;

}


.history-top {

    display: flex;

    justify-content: space-between;

    align-items: center;

    gap: 10px;

    flex-wrap: wrap;

}


.badge {

    padding: 6px 12px;

    border-radius: 20px;

    font-size: 13px;

    font-weight: bold;

}


.category {

    background: #e0e7ff;

    color: #3730a3;

}


.high {

    background: #fee2e2;

    color: #b91c1c;

}


.medium {

    background: #fef3c7;

    color: #92400e;

}


.low {

    background: #dcfce7;

    color: #166534;

}


.delete {

    background: #475569;

    color: white;

    padding: 7px 12px;

    font-size: 12px;

}


.empty {

    text-align: center;

    color: #94a3b8;

    padding: 20px;

}


@media(max-width:700px) {

    .result-grid {

        grid-template-columns: 1fr;

    }

    .header h1 {

        font-size: 25px;

    }

}


</style>

</head>


<body>


<div class="container">


    <!-- HEADER -->

    <div class="header">

        <h1>🎤 Speech-Based Complaint Analysis</h1>

        <p>
            Speak your complaint and let AI-style analysis
            categorize and prioritize it.
        </p>

    </div>


    <!-- INPUT -->

    <div class="card">

        <h2>📝 Submit Complaint</h2>


        <textarea
            id="complaint"
            placeholder="Speak your complaint or type it here..."
        ></textarea>


        <div class="buttons">

            <button
                class="record"
                onclick="startSpeech()">

                🎤 Start Recording

            </button>


            <button
                class="submit"
                onclick="submitComplaint()">

                📤 Submit Complaint

            </button>


            <button
                class="clear"
                onclick="clearText()">

                🗑 Clear

            </button>

        </div>


        <div
            id="status"
            class="status">

        </div>

    </div>


    <!-- RESULT -->

    <div
        class="card"
        id="resultCard"
        style="display:none;">

        <h2>📊 Complaint Analysis</h2>


        <div class="result-grid">


            <div class="result">

                <h3>COMPLAINT</h3>

                <p id="resultComplaint">
                </p>

            </div>


            <div class="result">

                <h3>CATEGORY</h3>

                <p id="resultCategory">
                </p>

            </div>


            <div class="result">

                <h3>PRIORITY</h3>

                <p id="resultPriority">
                </p>

            </div>


        </div>

    </div>


    <!-- HISTORY -->

    <div class="card">

        <h2>📋 Complaint History</h2>

        <div id="history">

            Loading complaints...

        </div>

    </div>


</div>


<script>


// ============================================================
// SPEECH RECOGNITION
// ============================================================

function startSpeech() {


    const SpeechRecognition =
        window.SpeechRecognition ||
        window.webkitSpeechRecognition;


    if (!SpeechRecognition) {

        alert(
            "Speech recognition is not supported. Please use Google Chrome."
        );

        return;

    }


    const recognition =
        new SpeechRecognition();


    recognition.lang = "en-IN";

    recognition.continuous = false;

    recognition.interimResults = false;


    document.getElementById("status")
        .innerText =
        "🎤 Listening... Please speak.";


    recognition.start();


    recognition.onresult = function(event) {

        const text =
            event.results[0][0].transcript;


        document.getElementById("complaint")
            .value = text;


        document.getElementById("status")
            .innerText =
            "✅ Speech converted to text.";

    };


    recognition.onerror = function(event) {

        document.getElementById("status")
            .innerText =
            "❌ Speech error: " + event.error;

    };


    recognition.onend = function() {

        console.log("Speech recognition ended.");

    };

}


// ============================================================
// SUBMIT COMPLAINT
// ============================================================

async function submitComplaint() {


    const text =
        document.getElementById("complaint")
            .value.trim();


    if (!text) {

        document.getElementById("status")
            .innerText =
            "⚠️ Please speak or type a complaint.";

        return;

    }


    document.getElementById("status")
        .innerText =
        "⏳ Analyzing complaint...";


    const response =
        await fetch("/add_complaint", {

            method: "POST",

            headers: {

                "Content-Type":
                    "application/json"

            },

            body: JSON.stringify({

                complaint: text

            })

        });


    const data =
        await response.json();


    if (data.success) {


        document.getElementById(
            "resultCard"
        ).style.display = "block";


        document.getElementById(
            "resultComplaint"
        ).innerText =
            data.complaint;


        document.getElementById(
            "resultCategory"
        ).innerText =
            data.category;


        document.getElementById(
            "resultPriority"
        ).innerText =
            data.priority;


        document.getElementById("status")
            .innerText =
            "✅ Complaint submitted successfully.";


        document.getElementById("complaint")
            .value = "";


        loadComplaints();


    } else {

        document.getElementById("status")
            .innerText =
            "❌ " + data.message;

    }

}


// ============================================================
// LOAD COMPLAINTS
// ============================================================

async function loadComplaints() {


    const response =
        await fetch("/complaints");


    const complaints =
        await response.json();


    const history =
        document.getElementById("history");


    if (complaints.length === 0) {

        history.innerHTML =
            '<div class="empty">No complaints submitted yet.</div>';

        return;

    }


    history.innerHTML = "";


    complaints.forEach(function(item) {


        let priorityClass =
            item.priority.toLowerCase();


        const div =
            document.createElement("div");


        div.className =
            "history-item";


        div.innerHTML = `

            <div class="history-top">

                <strong>
                    Complaint #${item.id}
                </strong>

                <div>

                    <span class="badge category">
                        ${item.category}
                    </span>

                    <span class="badge ${priorityClass}">
                        ${item.priority}
                    </span>

                    <button
                        class="delete"
                        onclick="deleteComplaint(${item.id})">

                        Delete

                    </button>

                </div>

            </div>


            <p style="margin-top:10px;">
                ${item.complaint}
            </p>


            <small style="color:#94a3b8;">
                ${item.created_at}
            </small>

        `;


        history.appendChild(div);

    });

}


// ============================================================
// DELETE COMPLAINT
// ============================================================

async function deleteComplaint(id) {


    const confirmDelete =
        confirm(
            "Are you sure you want to delete this complaint?"
        );


    if (!confirmDelete) {

        return;

    }


    await fetch(
        "/delete/" + id,
        {
            method: "DELETE"
        }
    );


    loadComplaints();

}


// ============================================================
// CLEAR TEXT
// ============================================================

function clearText() {

    document.getElementById("complaint")
        .value = "";

    document.getElementById("status")
        .innerText = "";

}


// ============================================================
// LOAD DATA WHEN PAGE OPENS
// ============================================================

loadComplaints();


</script>


</body>

</html>

"""


# ============================================================
# HOME ROUTE
# ============================================================

@app.route("/")
def home():

    return render_template_string(HTML)


# ============================================================
# ADD COMPLAINT
# ============================================================

@app.route(
    "/add_complaint",
    methods=["POST"]
)
def add_complaint():

    data = request.get_json()

    complaint = data.get(
        "complaint",
        ""
    ).strip()


    if not complaint:

        return jsonify({

            "success": False,

            "message":
            "Complaint cannot be empty."

        })


    category, priority =analyze_complaint(complaint)


    from datetime import datetime


    created_at = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )


    conn = sqlite3.connect(DATABASE)

    cursor = conn.cursor()


    cursor.execute("""
        INSERT INTO complaints
        (complaint, category, priority, created_at)
        VALUES (?, ?, ?, ?)
    """, (
        complaint,
        category,
        priority,
        created_at
    ))


    conn.commit()

    conn.close()


    return jsonify({

        "success": True,

        "complaint": complaint,

        "category": category,

        "priority": priority

    })


# ============================================================
# GET COMPLAINTS
# ============================================================

@app.route("/complaints")
def get_complaints():

    conn = sqlite3.connect(DATABASE)

    conn.row_factory = sqlite3.Row

    cursor = conn.cursor()


    cursor.execute("""
        SELECT *
        FROM complaints
        ORDER BY id DESC
    """)


    rows = cursor.fetchall()


    complaints = [

        dict(row)

        for row in rows

    ]


    conn.close()


    return jsonify(complaints)


# ============================================================
# DELETE COMPLAINT
# ============================================================

@app.route(
    "/delete/<int:complaint_id>",
    methods=["DELETE"]
)
def delete_complaint(complaint_id):


    conn = sqlite3.connect(DATABASE)

    cursor = conn.cursor()


    cursor.execute(
        "DELETE FROM complaints WHERE id = ?",
        (complaint_id,)
    )


    conn.commit()

    conn.close()


    return jsonify({

        "success": True

    })


# ============================================================
# INITIALIZE DATABASE
# ============================================================

init_db()


# ============================================================
# START FLASK
# ============================================================

def run_flask():

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(
    target=run_flask
)

thread.daemon = True

thread.start()


time.sleep(3)


# ============================================================
# GET COLAB WEB URL
# ============================================================

url = output.eval_js(
    "google.colab.kernel.proxyPort(5000)"
)


print("=" * 60)

print("🎤 SPEECH-BASED COMPLAINT ANALYSIS SYSTEM")

print("=" * 60)

print()

print("✅ Flask server started")

print("✅ SQLite database initialized")

print("✅ Speech recognition enabled")

print()

print("🌐 OPEN THIS LINK:")

print(url)

print()

print("=" * 60)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🎤 SPEECH-BASED COMPLAINT ANALYSIS SYSTEM

✅ Flask server started
✅ SQLite database initialized
✅ Speech recognition enabled

🌐 OPEN THIS LINK:
https://5000-m-s-kkb-euw4b0-3b19njpwovnjg-b.europe-west4-0.prod.colab.dev

